# Reconstructing Swing-Level Attack Angle / Direction from Batted-Ball Outcomes

**Goal:** test whether a per-swing "mishit proxy" (built from launch angle + exit velocity)
can explain real attack_angle / attack_direction, using MLB's tracked-swing ground truth
(2023-2025) as the target — and if it holds up, apply the fitted relationship to FL, which
has no bat-tracking.

**Scope decision for this version:** skip point-of-contact and pitch approach angle as
controls for now (both exist — MLB has `intercept_ball_minus_batter_pos_x/y_inches`, FL has
`contact_pos_x/y/z` and `vert_appr_angle`/`horz_appr_angle` directly — but adding them is a
v2 once the baseline mishit-proxy model is validated on its own).

**Design, in order:**
1. Fit a population-level EV-vs-LA envelope (same top-percentile-per-bin logic as
   `PAA`/`PAD_lac`) — this gives, for any launch angle, the expected *best possible* EV.
   `mishit_proxy = observed_EV - envelope_EV(LA)`. Non-circular: it predicts EV from LA,
   not LA from itself.
2. On MLB tracked swings, compare `attack_angle ~ launch_angle` (baseline) against
   `attack_angle ~ launch_angle + mishit_proxy` (augmented), held out by batter, so
   R² isn't inflated by the same swings training and testing. Same for `attack_direction`
   with spray angle.
3. If the augmented model beats baseline out-of-sample, apply the fitted MLB coefficients
   to FL — but recompute the mishit proxy using FL's *own* EV/LA envelope, since population
   baselines for what counts as a "mishit" plausibly differ between leagues even if the
   regression relationship transfers.
4. No FL ground truth exists to check step 3 against directly — that's the honest limit of
   this approach and is called out explicitly at the end, not glossed over.

## 1. Imports & Config

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from scipy.stats import pearsonr
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score

pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", "{:.3f}".format)

PULL_CUTOFF_DEG = 15
LA_MIN, LA_MAX = -30, 60
ENV_BIN_WIDTH   = 3
ENV_TOP_FRAC    = 0.15
MIN_BIN_N       = 5

print("Config OK")

## 2. Load MLB Statcast (tracked-swing subset)

Uses your actual pulled-column names — `game_year` (not `season`), `attack_angle` /
`attack_direction` / `bat_speed` as the real bat-tracking ground truth, `type == "X"` for
BBE, `hc_x`/`hc_y` for spray.

In [ ]:
def load_statcast(cache_path):
    p = Path(cache_path)
    if p.exists():
        print(f"Loading {p} ...")
        return pd.read_parquet(p)
    raise FileNotFoundError(f"{p} not found - build the cache first.")

raw24 = load_statcast("statcast_2024.parquet")
raw25 = load_statcast("statcast_2025.parquet")
raw = pd.concat([raw24, raw25], ignore_index=True)

mlb = raw[
    (raw["type"] == "X") &
    raw["launch_speed"].notna() &
    raw["launch_angle"].notna() &
    raw["hc_x"].notna() & raw["hc_y"].notna() &
    raw["stand"].notna() &
    raw["batter"].notna()
].copy()

mlb["spray_deg"] = np.degrees(
    np.arctan((mlb["hc_x"] - 125.42) / (198.27 - mlb["hc_y"]))
) * 0.75
mlb["spray_pull"] = np.where(mlb["stand"] == "R", -mlb["spray_deg"], mlb["spray_deg"])

n_tracked = mlb["attack_angle"].notna().sum()
print(f"MLB BBE: {len(mlb):,}   with attack_angle/direction ground truth: {n_tracked:,} "
      f"({n_tracked/len(mlb):.1%})")

## 3. Mishit Proxy — Population EV-vs-LA Envelope

Same top-percentile-per-bin + parabola-fit logic as `PAA`/`PAD_lac`, but fit once globally
(population-level, not per-batter) so it can be applied to every swing as a fixed reference
curve, and refit separately for FL later on FL's own population.

In [ ]:
def fit_ev_envelope(la, ev, la_min=LA_MIN, la_max=LA_MAX,
                    bin_width=ENV_BIN_WIDTH, top_frac=ENV_TOP_FRAC, min_bin_n=MIN_BIN_N):
    mask = (la >= la_min) & (la <= la_max) & ~np.isnan(la) & ~np.isnan(ev)
    la, ev = la[mask], ev[mask]
    bins = np.arange(la_min, la_max + bin_width, bin_width)
    bin_idx = np.digitize(la, bins)
    xs, ys = [], []
    for b in np.unique(bin_idx):
        m = bin_idx == b
        sub_la, sub_ev = la[m], ev[m]
        if len(sub_la) < min_bin_n:
            continue
        k = max(1, int(np.ceil(top_frac * len(sub_la))))
        top_idx = np.argpartition(sub_ev, -k)[-k:] if len(sub_ev) > k else np.arange(len(sub_ev))
        xs.append(sub_la[top_idx]); ys.append(sub_ev[top_idx])
    xs, ys = np.concatenate(xs), np.concatenate(ys)
    coefs = np.polyfit(xs, ys, deg=2)
    return coefs

def envelope_ev(la, coefs):
    return np.polyval(coefs, la)

mlb_env_coefs = fit_ev_envelope(mlb["launch_angle"].to_numpy(), mlb["launch_speed"].to_numpy())
mlb["envelope_ev"]  = envelope_ev(mlb["launch_angle"].to_numpy(), mlb_env_coefs)
mlb["mishit_proxy"] = mlb["launch_speed"] - mlb["envelope_ev"]

# sanity plot
fig, ax = plt.subplots(figsize=(7, 5))
sample = mlb.sample(min(20000, len(mlb)), random_state=0)
ax.scatter(sample["launch_angle"], sample["launch_speed"], s=2, alpha=0.15, color="#9E9E9E")
la_grid = np.linspace(LA_MIN, LA_MAX, 200)
ax.plot(la_grid, envelope_ev(la_grid, mlb_env_coefs), color="#D32F2F", lw=2, label="Fitted envelope")
ax.set_xlabel("Launch angle (deg)"); ax.set_ylabel("Exit velocity (mph)")
ax.set_title("MLB EV-vs-LA envelope (mishit_proxy = EV - this curve)")
ax.legend(); plt.tight_layout(); plt.show()

print(mlb["mishit_proxy"].describe().round(2))

## 4. Does the mishit proxy explain real attack_angle / attack_direction?

Held out by batter (`GroupKFold`) so a batter's swings never appear in both train and test —
otherwise R² is inflated by batter-level mean effects rather than reflecting real predictive
signal on new swings.

In [ ]:
GT = mlb[mlb["attack_angle"].notna() & mlb["attack_direction"].notna()].copy()
GT["spray_pull_signed"] = GT["spray_pull"]  # already handedness-corrected above
print(f"Tracked-swing rows for validation: {len(GT):,}   batters: {GT['batter'].nunique():,}")

def cv_r2(X_cols, y_col, df, n_splits=5):
    X = df[X_cols].to_numpy(dtype=float)
    y = df[y_col].to_numpy(dtype=float)
    groups = df["batter"].to_numpy()
    gkf = GroupKFold(n_splits=n_splits)
    preds = np.full(len(y), np.nan)
    for tr_idx, te_idx in gkf.split(X, y, groups):
        m = LinearRegression().fit(X[tr_idx], y[tr_idx])
        preds[te_idx] = m.predict(X[te_idx])
    return r2_score(y, preds), preds

print("=== attack_angle ===")
r2_base_aa, _ = cv_r2(["launch_angle"], "attack_angle", GT)
r2_aug_aa,  _ = cv_r2(["launch_angle", "mishit_proxy"], "attack_angle", GT)
print(f"  baseline (LA only):          held-out R^2 = {r2_base_aa:.4f}")
print(f"  augmented (LA + mishit):     held-out R^2 = {r2_aug_aa:.4f}")
print(f"  improvement:                 {r2_aug_aa - r2_base_aa:+.4f}")

print("\n=== attack_direction ===")
r2_base_ad, _ = cv_r2(["spray_pull_signed"], "attack_direction", GT)
r2_aug_ad,  _ = cv_r2(["spray_pull_signed", "mishit_proxy"], "attack_direction", GT)
print(f"  baseline (spray only):       held-out R^2 = {r2_base_ad:.4f}")
print(f"  augmented (spray + mishit):  held-out R^2 = {r2_aug_ad:.4f}")
print(f"  improvement:                 {r2_aug_ad - r2_base_ad:+.4f}")

In [ ]:
# Fit final models on the full tracked-swing set (for downstream FL application)
model_aa = LinearRegression().fit(GT[["launch_angle", "mishit_proxy"]], GT["attack_angle"])
model_ad = LinearRegression().fit(GT[["spray_pull_signed", "mishit_proxy"]], GT["attack_direction"])

print("attack_angle model:    ", dict(zip(["launch_angle", "mishit_proxy"], model_aa.coef_.round(3))),
      "intercept", round(model_aa.intercept_, 3))
print("attack_direction model:", dict(zip(["spray_pull_signed", "mishit_proxy"], model_ad.coef_.round(3))),
      "intercept", round(model_ad.intercept_, 3))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, col, model, X_cols, label in [
    (axes[0], "attack_angle", model_aa, ["launch_angle", "mishit_proxy"], "Attack angle"),
    (axes[1], "attack_direction", model_ad, ["spray_pull_signed", "mishit_proxy"], "Attack direction"),
]:
    pred = model.predict(GT[X_cols])
    ax.scatter(GT[col], pred, s=2, alpha=0.1, color="#1565C0")
    lims = [GT[col].min(), GT[col].max()]
    ax.plot(lims, lims, color="#D32F2F", lw=1, ls="--")
    ax.set_xlabel(f"True {label.lower()} (deg)"); ax.set_ylabel(f"Predicted {label.lower()} (deg)")
    ax.set_title(label)
plt.tight_layout(); plt.show()

**How to read the R² comparison above:** if the augmented model beats baseline by a
meaningful margin out-of-sample, the mishit proxy is carrying real signal about swing plane
beyond what raw LA/spray already gives you — worth carrying forward to FL. If the
improvement is negligible, LA/spray alone were already about as good a proxy as this
approach gets, and the added complexity isn't earning its keep — that's a legitimate
stopping point, not a failure to fix.

---
## 5. Apply to FL

FL has no `attack_angle`/`attack_direction` ground truth, so this step is **extrapolation,
not validation** — the R² check above is the only evidence this works, and it's MLB-only
evidence. The mishit proxy is recomputed on FL's own EV/LA population (population baselines
for "mishit" plausibly differ from MLB), but the regression coefficients themselves are
carried over unchanged from the MLB fit.

Swap the loader below for your actual Supabase pull — columns match what you listed
(`exit_speed`, `angle`, `direction`, `batter_side`, `pitch_call`, `batter_id`).

In [ ]:
def load_fl_bbe():
    """TODO: replace with your actual FLStuff Supabase query. Expected columns per
    your schema: batter_id, batter_side ('left'/'right'), exit_speed, angle, direction,
    pitch_call, game_id/year or similar for season grouping."""
    raise NotImplementedError("Wire up your Supabase connection here")

fl_raw = load_fl_bbe()

fl_bbe = fl_raw[
    (fl_raw["pitch_call"] == "in_play") &
    fl_raw["exit_speed"].notna() &
    fl_raw["angle"].notna() &
    fl_raw["direction"].notna() &
    fl_raw["batter_side"].notna()
].copy()

# direction is already a signed spray-type field in FL per your schema; confirm sign
# convention matches spray_pull_signed above (positive = pull) before trusting the model -
# if FL's `direction` isn't already pull-corrected by handedness, mirror it the same way
# spray_pull was built for mlb:
fl_bbe["spray_pull_signed"] = np.where(
    fl_bbe["batter_side"] == "right", -fl_bbe["direction"], fl_bbe["direction"]
)

fl_env_coefs = fit_ev_envelope(fl_bbe["angle"].to_numpy(), fl_bbe["exit_speed"].to_numpy())
fl_bbe["envelope_ev"]  = envelope_ev(fl_bbe["angle"].to_numpy(), fl_env_coefs)
fl_bbe["mishit_proxy"] = fl_bbe["exit_speed"] - fl_bbe["envelope_ev"]

fl_bbe["reconstructed_attack_angle"] = model_aa.predict(
    fl_bbe[["angle", "mishit_proxy"]].rename(columns={"angle": "launch_angle"})
)
fl_bbe["reconstructed_attack_direction"] = model_ad.predict(
    fl_bbe[["spray_pull_signed", "mishit_proxy"]]
)

print(f"FL BBE with reconstructed swing metrics: {len(fl_bbe):,}")
print(fl_bbe[["reconstructed_attack_angle", "reconstructed_attack_direction"]].describe().round(2))

**Limitations, stated plainly:**

- This is calibrated on MLB and applied to FL with no way to check per-swing accuracy in FL
  itself — treat `reconstructed_attack_angle`/`reconstructed_attack_direction` as a modeled
  estimate, not a measurement, when it reaches a hitter or coach.
- The mishit-proxy envelope is refit on FL's own population, but the underlying assumption —
  that the *relationship* between mishit quality and true attack angle is the same in FL as
  MLB — is untested. Talent level, bat/ball specs, and TrackMan calibration differences could
  all break that transfer silently.
- POC and pitch approach angle were deliberately excluded this round. If the baseline vs.
  augmented R² gap in §4 is small, adding those back in (both leagues have the fields) is the
  natural next experiment before concluding this approach has a ceiling.
- No uncertainty quantification on `reconstructed_attack_angle`/`_direction` yet — the same
  bootstrap + structural-decomposition approach from the PAD_lac/MAD notebook could be
  layered on here once/if this model is worth trusting further.